# 3. Model train + SHAP/FFA analysis

**Purpose:** Run pipeline (model data → PGx → final model), then SHAP and FFA analysis and combine results for the Causal tab. No build or deploy here.

**Flow:** Run this notebook first. Then [4_dashboard_visuals.ipynb](4_dashboard_visuals.ipynb) (BupaR, DTW, FP-Growth). Then [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb).

**Steps:** Sync inputs → Verify → Pipeline Phase 4 (model data) → Phase 5 (PGx) → Phase 6 (final model) → Step 1a (metadata) → Step 7 (SHAP) → Step 8 (FFA) → Combine (SHAP+FFA to dashboard) → optional inspection.

**Memory:** Pipeline scripts use **DuckDB and Parquet** where possible for efficient memory use (Step 4 model data, Step 6/7 SHAP/FFA data prep, combine); pandas is used only where required (e.g. model/SHAP APIs). See project `.cursorrules` for data-processing preferences.\n\nPrerequisites: Cohorts (Step 2), feature importance (Step 3/3b). Run from repo root.

In [17]:
# Setup: paths and project root
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "10_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "10_risk_dashboard").exists():
    pass
else:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import sync_s3_to_local, check_step_checkpoint_exists, save_step_checkpoint

DASHBOARD_DIR = PROJECT_ROOT / "10_risk_dashboard"
DATA_PREP_DIR = DASHBOARD_DIR / "data_preparation"
DEPLOY_DIR = DASHBOARD_DIR / "deployment"
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

print("PGx Risk Calculator Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dashboard dir: {DASHBOARD_DIR}")
print(f"Data prep: {DATA_PREP_DIR}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print("=" * 60)

PGx Risk Calculator Workflow
Project root: /home/pgx3874/pgx-analysis
Dashboard dir: /home/pgx3874/pgx-analysis/10_risk_dashboard
Data prep: /home/pgx3874/pgx-analysis/10_risk_dashboard/data_preparation
Data root (NVMe/local): /mnt/nvme


In [18]:
# Both cohorts use full age band set
from py_helpers.constants import REQUIRED_COHORTS

# Input dirs (required for pipeline Step 4–6)
# Cohorts: Step 2 cohort.parquet files (create_model_data reads case/control and target dates from here).
COHORTS_ROOT = DATA_ROOT / "gold" / "cohorts"
# Feature importance: Step 3/3b outputs — cohort_feature_importance.csv and feature_filtering_summary.json per cohort/age_band.
FI_ROOT = DATA_ROOT / "gold" / "feature_importance"
STEP3_OUTPUTS = STEP3B_OUTPUTS = FI_ROOT
# Model data: single canonical location (Step 4 output, Step 5/6 input).
from py_helpers.env_utils import get_model_data_root
MODEL_DATA_ROOT = get_model_data_root()

# Output dirs (Step 6 final model outputs; data prep and Lambda read from these)
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_OUTPUTS_ALT = DATA_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_GOLD = DATA_ROOT / "gold" / "final_model"  # S3 layout: cohort/13-24/*.joblib

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("\nInput dirs (for Step 4–6):")
print(f"  Cohorts (2):              {COHORTS_ROOT}")
print(f"  Feature importance (3/3b): {FI_ROOT}  (CSVs + feature_filtering_summary.json)")
print(f"  Model data (4; in/out):   {MODEL_DATA_ROOT}")
print("\nOutput dirs (Step 6):")
print(f"  Project:   {FINAL_MODEL_OUTPUTS}")
print(f"  NVMe:      {FINAL_MODEL_OUTPUTS_ALT}")
print(f"  gold/NVMe: {FINAL_MODEL_GOLD}")

Cohorts and age bands:
  opioid_ed: ['0-12', '13-24', '25-44', '45-54', '55-64', '65-74', '75-84', '85-114']
  non_opioid_ed: ['0-12', '13-24', '25-44', '45-54', '55-64', '65-74', '75-84', '85-114']

Input dirs (for Step 4–6):
  Cohorts (2):              /mnt/nvme/gold/cohorts
  Feature importance (3/3b): /mnt/nvme/gold/feature_importance  (CSVs + feature_filtering_summary.json)
  Model data (4; in/out):   /mnt/nvme/4_model_data

Output dirs (Step 6):
  Project:   /home/pgx3874/pgx-analysis/6_final_model/outputs
  NVMe:      /mnt/nvme/6_final_model/outputs
  gold/NVMe: /mnt/nvme/gold/final_model


## Clear all checkpoints and pipeline outputs (optional — for a fresh run)

Run this cell **once** when you want to rebuild the full pipeline from Step 4 through SHAP/FFA from scratch. It (1) clears S3 **checkpoints** (pgx-repository: 4_model_data, 6_final_model, 9_dashboard_metadata), (2) deletes S3 **pipeline outputs** (pgxdatalake: `gold/cohorts_model_data/`, `gold/final_model/`) so Step 4 and Step 6 re-run instead of re-downloading, (3) removes local output directories. After this, run the Sync cell and then Steps 4 → 5 → 6 → 1a → Step 7 → Step 8 → Combine.

In [ ]:
# Clear S3 checkpoints, S3 pipeline outputs, and local outputs for a fresh model + SHAP/FFA run.
import shutil
import subprocess
from py_helpers.workflow_sync_checkpoint import clear_step_checkpoints, delete_step_checkpoint

# 1) S3 checkpoint metadata (pgx-repository) so steps don't think they're done
for step in ("4_model_data", "6_final_model"):
    for cohort, bands in REQUIRED_COHORTS.items():
        n = clear_step_checkpoints(step, cohort, bands, logger=None)
        print(f"Cleared {n} checkpoint(s) for {step} / {cohort}")
delete_step_checkpoint("9_dashboard_metadata", "all", "all", logger=None)
print("Cleared checkpoint 9_dashboard_metadata (all/all)")

# 2) S3 pipeline outputs (pgxdatalake) so Step 4 and Step 6 re-run instead of re-downloading
_aws = shutil.which("aws") or "aws"
_profile = ["--profile", AWS_PROFILE] if AWS_PROFILE else []
for prefix in ("gold/cohorts_model_data/", "gold/final_model/"):
    uri = f"s3://{S3_BUCKET}/{prefix}"
    r = subprocess.run([_aws, "s3", "rm", uri, "--recursive"] + _profile, capture_output=True, text=True)
    if r.returncode == 0:
        print(f"Cleared S3 {uri}")
    else:
        print(f"S3 rm {uri}: exit {r.returncode} (check credentials); {r.stderr or r.stdout or ''}")

# 3) Local output directories
dirs_to_clear = [
    MODEL_DATA_ROOT,
    FINAL_MODEL_OUTPUTS,
    FINAL_MODEL_OUTPUTS_ALT,
    PROJECT_ROOT / "7_shap_analysis" / "outputs",
    PROJECT_ROOT / "8_ffa_analysis" / "outputs",
    PROJECT_ROOT / "10_risk_dashboard" / "outputs",
]
for d in dirs_to_clear:
    d = Path(d)
    if d.exists():
        shutil.rmtree(d)
        print(f"Removed {d}")
    else:
        print(f"(skip, not present) {d}")
print("Done. Re-run Sync and then Steps 4 → 5 → 6 → 1a → Step 7 → Step 8 → Combine for a fresh model and SHAP/FFA outputs.")

## Sync required inputs from S3 to NVMe (idempotent)

Sync **cohorts** (Step 2), **feature importance** (Step 3/3b), and **Step 6** final model outputs from S3 so pipeline and data preparation can read from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [ ]:
# Sync cohorts (Step 2), Step 3a/3b feature importance, and Step 6 final models from S3 to NVMe (DATA_ROOT).
# Cohorts -> COHORTS_ROOT (gold/cohorts); Feature importance -> gold/feature_importance; Step 6 -> gold/final_model.
COHORTS_ROOT.mkdir(parents=True, exist_ok=True)
FI_SYNC_TARGET = DATA_ROOT / "gold" / "feature_importance"
FI_SYNC_TARGET.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_GOLD.mkdir(parents=True, exist_ok=True)

sync_s3_to_local(f"s3://{S3_BUCKET}/gold/cohorts/", COHORTS_ROOT, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/feature_importance/", FI_SYNC_TARGET, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/final_model/", FINAL_MODEL_GOLD, profile=AWS_PROFILE)
print("Sync complete. Run Step 0 verification below.")

## Step 0: Verify inputs (FI required; 4_model_data and Step 6 informational)

**Required:** **Feature importance** (Step 3/3b) — must exist for each cohort/age_band so Pipeline Step 4 can run.

**Informational:** **ModelData** checks `DATA_ROOT/4_model_data` and `PROJECT_ROOT/4_model_data` (same location `create_model_data.py` writes to). **Model** = Step 6 outputs. Both are produced by Pipeline Step 4–6 cells below; if already present, you can skip those cells.

In [ ]:
def check_feature_importance(cohort: str, age_band: str) -> bool:
    """Check if feature importance exists using FileResolver pattern."""
    from py_helpers.file_resolver import FileResolver
    # Check Step 3b refined cohort feature importance first
    resolver_3b = FileResolver(
        file_type="cohort_feature_importance",
        project_root=PROJECT_ROOT,
        cohort=cohort,
        age_band=age_band,
        auto_download=False
    )
    if resolver_3b.exists():
        return True
    # Fallback to Step 3a aggregated feature importance
    resolver_3a = FileResolver(
        file_type="aggregated_feature_importance",
        project_root=PROJECT_ROOT,
        cohort=cohort,
        age_band=age_band,
        auto_download=False
    )
    return resolver_3a.exists()

def check_cohorts(cohort: str, age_band: str) -> bool:
    """Check Step 2 cohort.parquet exists for at least one year (2016–2019). Layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet."""
    for year in (2016, 2017, 2018, 2019):
        p = COHORTS_ROOT / f"cohort_name={cohort}" / f"event_year={year}" / f"age_band={age_band}" / "cohort.parquet"
        if p.exists():
            return True
    return False

def check_model_data(cohort: str, age_band: str) -> bool:
    """Check model_events.parquet at canonical MODEL_DATA_ROOT (same location create_model_data.py writes to)."""
    p = MODEL_DATA_ROOT / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
    return p.exists()

def check_final_model(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # 1) Project or DATA_ROOT/6_final_model/outputs: cohort/13_24/models/*.joblib
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        model_dir = base / cohort / ab
        if not model_dir.exists():
            continue
        models_sub = model_dir / "models"
        if models_sub.exists() and any(models_sub.glob("*.joblib")):
            return True
        if (model_dir / "feature_schema.json").exists():
            return True
    # 2) DATA_ROOT/gold/final_model (S3-synced): cohort/13-24/*.joblib (hyphen in age_band)
    gold_dir = FINAL_MODEL_GOLD / cohort / age_band
    if gold_dir.exists() and any(gold_dir.glob("*.joblib")):
        return True
    return False

print("Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)")
print("  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs")
fi_ok_all = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        cohorts_ok = check_cohorts(cohort, age_band)
        fi_ok = check_feature_importance(cohort, age_band)
        model_data_ok = check_model_data(cohort, age_band)
        model_ok = check_final_model(cohort, age_band)
        if not fi_ok:
            fi_ok_all = False
        status = "ready" if fi_ok else "missing FI"
        print(f"  {cohort} / {age_band}:  Cohorts={cohorts_ok}, FI={fi_ok}, ModelData={model_data_ok}, Model={model_ok}  -> {status}")
if fi_ok_all:
    print("\nAll prerequisites are available to build model data. Run Pipeline Step 4–6 cells below.")
    print("  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)")
else:
    print("\nMissing feature importance for some cohort/age_band. Sync from S3 or run Step 3/3b first, then re-run this cell.")
if fi_ok_all:
    cohorts_missing = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands if not check_cohorts(c, ab)]
    if cohorts_missing:
        print("\nCohorts=False for some cohort/age_band. Sync gold/cohorts from S3 (run Sync cell) or run Step 2. Expected layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016–2019).")


# Pipeline Phase 4: Model data

Build `model_events.parquet` for each cohort/age_band from Step 2 cohort data and Step 3b feature importance. Outputs go to `MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet`. Run the cell below for all PGx cohorts/age_bands defined in this notebook.

In [ ]:
# Pipeline Step 4: BUILD model_events.parquet by running create_model_data.py, then QA.
# The script READS: COHORTS_ROOT (cohort.parquet), gold/medical, gold/pharmacy, and feature importance.
# It WRITES: MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet
import duckdb

def _model_data_candidates(cohort: str, age_band: str):
    """Canonical location for model_events.parquet (Step 4 writes to MODEL_DATA_ROOT)."""
    return [MODEL_DATA_ROOT]

def _model_data_path(cohort: str, age_band: str) -> Path:
    """Resolve model_events.parquet path (Step 4 writes to get_model_data_root() = DATA_ROOT or PROJECT on Linux)."""
    for base in _model_data_candidates(cohort, age_band):
        p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
        if p.exists():
            return p
    return None

def _log_model_data_qa(cohort: str, age_band: str) -> None:
    """Log location, target distribution, and control:case ratio for model_events.parquet."""
    path = _model_data_path(cohort, age_band)
    if not path:
        print(f"  [WARN] model_events.parquet not found for {cohort}/{age_band}")
        for base in _model_data_candidates(cohort, age_band):
            p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
            print(f"    Checked: {p}  (exists: {p.exists()})")
        print(f"    Build did not write output. Check script stdout above: [INFO] data roots and example cohort path (exists=?). Layout must be {COHORTS_ROOT}/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016–2019). Sync cohorts to COHORTS_ROOT if needed, then re-run this cell.")
        return
    print(f"  Location: {path}")
    con = duckdb.connect()
    try:
        dist = con.execute("SELECT target, COUNT(*)::BIGINT AS n FROM read_parquet(?) GROUP BY target ORDER BY target", [str(path)]).fetchall()
        total = sum(row[1] for row in dist)
        by_target = {int(row[0]): int(row[1]) for row in dist}
        n_controls = by_target.get(0, 0)
        n_cases = by_target.get(1, 0)
        ratio = (n_controls / n_cases) if n_cases else 0
        print(f"  Target distribution: {by_target} (total rows: {total:,})")
        print(f"  Control:case ratio: {n_controls:,}:{n_cases:,} = {ratio:.2f}:1")
    finally:
        con.close()

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 4: {cohort} / {age_band} (building model_events.parquet)")
        r = subprocess.run(
            [sys.executable, "create_model_data.py", "--cohort", cohort, "--age-band", age_band],
            cwd=PROJECT_ROOT / "4_model_data",
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
        _log_model_data_qa(cohort, age_band)
print("Step 4 complete.")

# Pipeline Phase 5: PGx analysis

Add PGx features (e.g. CPIC drug counts) to model data. Reads from Step 4 outputs and writes updated model data used by Step 6. Run for each cohort/age_band.

In [ ]:
# Pipeline Step 5: run_analysis.py for each REQUIRED_COHORTS (cohort, age_band)
# Set FORCE_STEP5 = True to re-run even when S3 outputs or checkpoints exist
FORCE_STEP5 = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 5: {cohort} / {age_band}")
        cmd = [sys.executable, "run_analysis.py", "--cohort-name", cohort, "--age-band", age_band]
        if FORCE_STEP5:
            cmd.append("--force")
        r = subprocess.run(cmd, cwd=PROJECT_ROOT / "5_pgx_analysis")
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 5 complete.")

# Pipeline Phase 6: Final model deployment outputs

Train final models per cohort/age_band. Reads Step 4 model data and Step 5 PGx features; writes trained models and `feature_schema.json` to `6_final_model/outputs` (or DATA_ROOT). When Optuna is available, Step 6 runs multi-objective HPO (Recall + AUC-PR) then 25-split MCCV with the best config; otherwise it uses fixed hyperparameters. Training uses `n_jobs` = number of CPU cores. Saves XGBoost and CatBoost feature importances from the final (full-data) models. These outputs are used by "Prepare models" and deployment below.

In [ ]:
# Pipeline Step 6: run_final_model.py for each REQUIRED_COHORTS (cohort, age_band)
# Note: script uses --age_band (underscore)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 6: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "run_final_model.py", "--cohort", cohort, "--age_band", age_band],
            cwd=PROJECT_ROOT / "6_final_model",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 6 complete.")

### Model performance summary and Top 20 feature importance (per cohort)

Before shutting down EC2: print final model performance metrics for all cohorts and Top 20 XGBoost feature importance bar charts per cohort.

In [ ]:
# Model performance metrics (all cohorts) and Top 20 feature importance bar charts per cohort
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Resolve outputs base (project or NVMe)
def _outputs_base():
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        if base and base.exists():
            return base
    return FINAL_MODEL_OUTPUTS

base = _outputs_base()
print("Final model performance by cohort (selected model)")
print("=" * 80)
all_metrics = []
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        path = base / cohort / ab_f / f"{cohort}_{ab_f}_model_metrics_summary.csv"
        if not path.exists():
            print(f"  [skip] {cohort} / {age_band}: no metrics CSV")
            continue
        df = pd.read_csv(path)
        selected = df.loc[df["selected"] == True]
        if selected.empty:
            selected = df.head(1)
        for _, row in selected.iterrows():
            all_metrics.append({
                "cohort": cohort,
                "age_band": age_band,
                "model": row["model"],
                "recall_mean": row["recall_mean"],
                "pr_auc_mean": row["pr_auc_mean"],
                "auc_mean": row.get("auc_mean", None),
                "logloss_mean": row.get("logloss_mean", None),
            })
if all_metrics:
    summary = pd.DataFrame(all_metrics)
    print(summary.to_string(index=False))
else:
    print("  No metrics CSVs found under", base)
print()
print("Top 20 feature importance (XGBoost) per cohort")
print("=" * 80)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        fi_path = base / cohort / ab_f / f"{cohort}_{ab_f}_xgboost_feature_importance.csv"
        if not fi_path.exists():
            print(f"  [skip] {cohort} / {age_band}: no feature importance CSV")
            continue
        fi = pd.read_csv(fi_path).sort_values("importance", ascending=False).head(20)
        if fi.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(range(len(fi)), fi["importance"].values, align="center")
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi["feature"].values, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("Importance (gain)")
        ax.set_title(f"Top 20 features — {cohort} / {age_band}")
        plt.tight_layout()
        plt.show()


### Step 1a: Generate Model Metadata

Extract valid codes (drugs, ICD, CPT) from feature importance for dashboard dropdowns. Uses Step 3b `cohort_feature_importance` when available, else Step 3 aggregated. **Checkpoint:** step is skipped if S3 checkpoint exists. Set `FORCE_STEP1A = True` in the cell below to re-run and rebuild `outputs/metadata/` (e.g. for Prepare Lambda directory). Run this before Step 7 (SHAP) so metadata is ready for deployment.

In [ ]:
import logging
logger = logging.getLogger(__name__)
FORCE_STEP1A = True  # Set True to re-run and build outputs/metadata even when checkpoint exists
if not FORCE_STEP1A and check_step_checkpoint_exists("9_dashboard_metadata", "all", "all", logger):
    print("Step 1 (generate metadata) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "generate_metadata.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_metadata", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

### Step 7: SHAP values

Generate SHAP values for each cohort/age_band using **7_shap_analysis/run_shap_analysis.py**. Outputs go to `7_shap_analysis/outputs/{cohort}/{age_band_fname}/` (global importance CSV, sample parquet). Run this **before** Step 8 (FFA) and Combine.

In [ ]:
# Step 7: Generate SHAP values per cohort/age_band (7_shap_analysis/outputs).
SHAP_SCRIPT = PROJECT_ROOT / "7_shap_analysis" / "run_shap_analysis.py"
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 7 (SHAP): {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, str(SHAP_SCRIPT), "--cohort", cohort, "--age_band", age_band],
            cwd=PROJECT_ROOT,
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 7 (SHAP) complete.")

### Step 8: FFA rules

Run FFA (Formal Feature Attribution) per cohort/age_band using **run_shap_ffa_workflow.py** with **--skip-shap --skip-combine**: uses existing Step 7 SHAP and XGBoost JSON, writes to `8_ffa_analysis/outputs`. Run this **after** Step 7 and **before** Combine.

In [ ]:
# Step 8: Generate FFA rules per cohort/age_band (8_ffa_analysis/outputs). Uses existing SHAP; no combine yet.
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 8 (FFA): {cohort} / {age_band}")
        r = subprocess.run(
            [
                sys.executable, "run_shap_ffa_workflow.py",
                "--cohort", cohort, "--age-band", age_band,
                "--skip-shap", "--skip-combine",
            ],
            cwd=DATA_PREP_DIR,
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 8 (FFA) complete.")

### Combine: SHAP + FFA → dashboard outputs

Run **combine_shap_ffa_results.py** per cohort/age_band to produce `10_risk_dashboard/visualizations/causal/{cohort}/{age_band_fname}/` (dashboard_data.json, combined_importance.csv, top_causal_factors, etc.) for the Causal Analysis tab and upload_causal_outputs_to_s3. Requires Step 7 and Step 8 outputs. Use `--workers 0` for auto worker count or `--workers 1` for sequential.

In [19]:
# Combine: Merge SHAP + FFA per cohort/age_band into 10_risk_dashboard/visualizations/causal
# (dashboard_data.json and combined_importance.csv for Causal tab and upload_causal_outputs_to_s3)
CAUSAL_VISUALS = PROJECT_ROOT / "10_risk_dashboard" / "visualizations" / "causal"
COMBINE_SCRIPT = DATA_PREP_DIR / "combine_shap_ffa_results.py"
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Combine: {cohort} / {age_band}")
        r = subprocess.run(
            [
                sys.executable, str(COMBINE_SCRIPT),
                "--cohort", cohort, "--age-band", age_band,
                "--output-dir", str(CAUSAL_VISUALS), "--workers", "0",
            ],
            cwd=DATA_PREP_DIR,
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Combine complete.")

→ Combine: opioid_ed / 0-12


2026-02-27 13:45:51,747 - INFO - Combining SHAP and FFA results for opioid_ed / 0-12
2026-02-27 13:45:51,747 - INFO - Found SHAP results: /home/pgx3874/pgx-analysis/7_shap_analysis/outputs/opioid_ed/0_12/opioid_ed_0_12_shap_global_importance_xgboost.csv
2026-02-27 13:45:51,747 - INFO - Found FFA explanations: /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs/opioid_ed/0_12/xgboost/axp_explanations.parquet
2026-02-27 13:45:51,747 - INFO - Found FFA importance: /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs/opioid_ed/0_12/xgboost/feature_importance_axp.parquet
2026-02-27 13:45:51,747 - INFO - All required inputs found: SHAP importance, SHAP sample, FFA explanations, FFA importance
2026-02-27 13:45:51,811 - INFO - Loaded SHAP sample values for patient explanations: shape (173, 54)
2026-02-27 13:45:51,817 - INFO - Patient explanations: 173 patients, 29 chunks, 31 workers
2026-02-27 13:45:51,895 - INFO - Saved consensus features to /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizati


SHAP + FFA COMBINED ANALYSIS SUMMARY
  Cohort: opioid_ed / 0-12

FEATURE TYPES (combined importance):
  drug: 1, icd: 8, cpt: 43, other: 2
  Expected: Drug + ICD + CPT  [OK]

CONSENSUS FEATURES:
  - Consensus features: 19
  - SHAP-only features: 1
  - FFA-only features: 1
  - Consensus rate: 95.0%

  High-confidence features (consensus):
    - item_cpt_81003
    - item_cpt_87880
    - item_cpt_90791
    - item_cpt_90837
    - item_cpt_92015
    - item_cpt_92551
    - item_cpt_96110
    - item_cpt_99213
    - item_cpt_99214
    - item_cpt_99215

COMBINED FEATURE IMPORTANCE (Top 10):
  1. n_events: 1.0000 (SHAP: 1.000, FFA: 1.000)
  2. pgx_num_drugs: 0.2871 (SHAP: 0.438, FFA: 0.136)
  3. item_cpt_99214: 0.2096 (SHAP: 0.372, FFA: 0.047)
  4. item_icd_Z23: 0.1665 (SHAP: 0.312, FFA: 0.021)
  5. item_cpt_92015: 0.1552 (SHAP: 0.278, FFA: 0.032)
  6. item_cpt_99393: 0.1547 (SHAP: 0.281, FFA: 0.028)
  7. item_cpt_87880: 0.1120 (SHAP: 0.210, FFA: 0.014)
  8. item_cpt_90837: 0.0861 (SHAP: 0.161,

2026-02-27 13:45:52,266 - INFO - Combining SHAP and FFA results for opioid_ed / 13-24
2026-02-27 13:45:52,267 - INFO - Found SHAP results: /home/pgx3874/pgx-analysis/7_shap_analysis/outputs/opioid_ed/13_24/opioid_ed_13_24_shap_global_importance_xgboost.csv
2026-02-27 13:45:52,267 - INFO - Found FFA explanations: /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs/opioid_ed/13_24/xgboost/axp_explanations.parquet
2026-02-27 13:45:52,267 - INFO - Found FFA importance: /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs/opioid_ed/13_24/xgboost/feature_importance_axp.parquet
2026-02-27 13:45:52,267 - INFO - All required inputs found: SHAP importance, SHAP sample, FFA explanations, FFA importance
2026-02-27 13:45:52,535 - INFO - Loaded SHAP sample values for patient explanations: shape (13710, 382)
2026-02-27 13:45:52,541 - INFO - Patient explanations: 2000 patients, 31 chunks, 31 workers
2026-02-27 13:45:53,976 - INFO - Saved consensus features to /home/pgx3874/pgx-analysis/10_risk_dashboard/vi


SHAP + FFA COMBINED ANALYSIS SUMMARY
  Cohort: opioid_ed / 13-24

FEATURE TYPES (combined importance):
  drug: 48, icd: 131, cpt: 201, other: 2
  Expected: Drug + ICD + CPT  [OK]

CONSENSUS FEATURES:
  - Consensus features: 19
  - SHAP-only features: 1
  - FFA-only features: 1
  - Consensus rate: 95.0%

  High-confidence features (consensus):
    - item_cpt_36415
    - item_cpt_80053
    - item_cpt_90651
    - item_cpt_93010
    - item_cpt_96372
    - item_cpt_99213
    - item_cpt_99283
    - item_cpt_99284
    - item_cpt_99285
    - item_cpt_99394

COMBINED FEATURE IMPORTANCE (Top 10):
  1. n_events: 0.9841 (SHAP: 0.968, FFA: 1.000)
  2. pgx_num_drugs: 0.8248 (SHAP: 1.000, FFA: 0.650)
  3. item_cpt_99394: 0.1392 (SHAP: 0.255, FFA: 0.023)
  4. item_drug_GABAPENTIN: 0.0670 (SHAP: 0.116, FFA: 0.018)
  5. item_cpt_99284: 0.0549 (SHAP: 0.101, FFA: 0.009)
  6. item_icd_Z23: 0.0535 (SHAP: 0.102, FFA: 0.005)
  7. item_cpt_99213: 0.0513 (SHAP: 0.094, FFA: 0.009)
  8. item_cpt_80053: 0.0464 (S

...

2026-02-27 13:46:57,283 - INFO - Combining SHAP and FFA results for non_opioid_ed / 75-84
2026-02-27 13:46:57,283 - INFO - Found SHAP results: /home/pgx3874/pgx-analysis/7_shap_analysis/outputs/non_opioid_ed/75_84/non_opioid_ed_75_84_shap_global_importance_xgboost.csv
2026-02-27 13:46:57,284 - INFO - Found FFA explanations: /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs/non_opioid_ed/75_84/xgboost/axp_explanations.parquet
2026-02-27 13:46:57,284 - INFO - Found FFA importance: /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs/non_opioid_ed/75_84/xgboost/feature_importance_axp.parquet
2026-02-27 13:46:57,284 - INFO - All required inputs found: SHAP importance, SHAP sample, FFA explanations, FFA importance
2026-02-27 13:46:57,344 - INFO - Loaded SHAP sample values for patient explanations: shape (3193, 7)
2026-02-27 13:46:57,349 - INFO - Patient explanations: 2000 patients, 31 chunks, 31 workers
2026-02-27 13:46:57,468 - INFO - Saved consensus features to /home/pgx3874/pgx-analysis/10_


SHAP + FFA COMBINED ANALYSIS SUMMARY
  Cohort: non_opioid_ed / 75-84

FEATURE TYPES (combined importance):
  drug: 5, icd: 0, cpt: 0, other: 2
  Expected: Drug only  [OK]

CONSENSUS FEATURES:
  - Consensus features: 7
  - SHAP-only features: 0
  - FFA-only features: 0
  - Consensus rate: 35.0%

  High-confidence features (consensus):
    - item_drug_CEPHALEXIN
    - item_drug_FUROSEMIDE
    - item_drug_LISINOPRIL
    - item_drug_MIRTAZAPINE
    - item_drug_OMEPRAZOLE
    - n_events
    - pgx_num_drugs

COMBINED FEATURE IMPORTANCE (Top 10):
  1. n_events: 1.0000 (SHAP: 1.000, FFA: 1.000)
  2. pgx_num_drugs: 0.2309 (SHAP: 0.259, FFA: 0.203)
  3. item_drug_OMEPRAZOLE: 0.0012 (SHAP: 0.002, FFA: 0.000)
  4. item_drug_FUROSEMIDE: 0.0003 (SHAP: 0.001, FFA: 0.000)
  5. item_drug_LISINOPRIL: 0.0002 (SHAP: 0.000, FFA: 0.000)
  6. item_drug_MIRTAZAPINE: 0.0002 (SHAP: 0.000, FFA: 0.000)
  7. item_drug_CEPHALEXIN: 0.0000 (SHAP: 0.000, FFA: 0.000)

PATIENT EXPLANATIONS:
  - Total patients analyzed:

2026-02-27 13:46:57,846 - INFO - Combining SHAP and FFA results for non_opioid_ed / 85-114
2026-02-27 13:46:57,847 - INFO - Found SHAP results: /home/pgx3874/pgx-analysis/7_shap_analysis/outputs/non_opioid_ed/85_114/non_opioid_ed_85_114_shap_global_importance_xgboost.csv
2026-02-27 13:46:57,847 - INFO - Found FFA explanations: /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs/non_opioid_ed/85_114/xgboost/axp_explanations.parquet
2026-02-27 13:46:57,847 - INFO - Found FFA importance: /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs/non_opioid_ed/85_114/xgboost/feature_importance_axp.parquet
2026-02-27 13:46:57,847 - INFO - All required inputs found: SHAP importance, SHAP sample, FFA explanations, FFA importance
2026-02-27 13:46:57,907 - INFO - Loaded SHAP sample values for patient explanations: shape (2523, 11)
2026-02-27 13:46:57,913 - INFO - Patient explanations: 2000 patients, 31 chunks, 31 workers
2026-02-27 13:46:58,031 - INFO - Saved consensus features to /home/pgx3874/pgx-analys


SHAP + FFA COMBINED ANALYSIS SUMMARY
  Cohort: non_opioid_ed / 85-114

FEATURE TYPES (combined importance):
  drug: 9, icd: 0, cpt: 0, other: 2
  Expected: Drug only  [OK]

CONSENSUS FEATURES:
  - Consensus features: 11
  - SHAP-only features: 0
  - FFA-only features: 0
  - Consensus rate: 55.0%

  High-confidence features (consensus):
    - item_drug_AMOXICILLIN
    - item_drug_AZITHROMYCIN
    - item_drug_CEPHALEXIN
    - item_drug_FUROSEMIDE
    - item_drug_LATANOPROST
    - item_drug_LISINOPRIL
    - item_drug_OMEPRAZOLE
    - item_drug_PREDNISONE
    - item_drug_SIMVASTATIN
    - n_events

COMBINED FEATURE IMPORTANCE (Top 10):
  1. n_events: 1.0000 (SHAP: 1.000, FFA: 1.000)
  2. pgx_num_drugs: 0.0757 (SHAP: 0.082, FFA: 0.069)
  3. item_drug_FUROSEMIDE: 0.0057 (SHAP: 0.011, FFA: 0.000)
  4. item_drug_AZITHROMYCIN: 0.0035 (SHAP: 0.007, FFA: 0.000)
  5. item_drug_CEPHALEXIN: 0.0023 (SHAP: 0.005, FFA: 0.000)
  6. item_drug_SIMVASTATIN: 0.0015 (SHAP: 0.003, FFA: 0.000)
  7. item_drug_

2026-02-27 13:46:58,058 - INFO - Saved patient explanations to /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/causal/non_opioid_ed/85_114/patient_explanations.csv
2026-02-27 13:46:58,059 - INFO - Saved summary report to /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/causal/non_opioid_ed/85_114/summary_report.txt


### Review combined SHAP/FFA and metadata

**Code set by design:**
- **opioid_ed**: Drug + ICD + CPT (all three used for Causal tab dropdowns).
- **non_opioid_ed**: Drug only (no ICD/CPT).

**How this aligns to output:** The pipeline enforces this before model and dashboard outputs.
- **Step 3a** (`py_helpers/feature_importance_utils.py`): For `non_opioid_ed` and age band ≥65, the feature palette is restricted to `drug_name` only, so aggregated feature importance (and thus Step 3b refined FI) contains only drug features.
- **Step 4**: Uses Step 3b feature list to build `model_events.parquet`; for non_opioid_ed that list is drug-only.
- **Step 6** (`6_final_model/run_final_model.py`): For non_opioid_ed, any `item_icd_*` and `item_cpt_*` columns are explicitly removed from the final feature matrix (polypharmacy = drugs only).
- **Step 1a** (generate_metadata): Reads the same feature importance CSVs; opioid_ed gets drugs + ICDs + CPTs, non_opioid_ed gets drugs only (and 0 CPT / few ICD in metadata is expected).

Run the cell below after **Combine** to verify metadata code counts and that combined importance has the expected columns. Dashboard/Lambda use **metadata** (Step 1a) for Drug/ICD/CPT lists; the combine script outputs `feature` names and scores only.

In [20]:
import json
import pandas as pd
from pathlib import Path

DASHBOARD_OUT = PROJECT_ROOT / "10_risk_dashboard" / "outputs"
CAUSAL_VISUALS = PROJECT_ROOT / "10_risk_dashboard" / "visualizations" / "causal"
META_DIR = DASHBOARD_OUT / "metadata"

DRUG_PREFIX = "item_drug_"
ICD_PREFIX  = "item_icd_"
CPT_PREFIX  = "item_cpt_"

def check_cohort_expectations(
    cohort: str,
    n_drugs: int, n_icds: int, n_cpts: int,
    n_drug_f: int, n_icd_f: int, n_cpt_f: int
) -> tuple[bool, str]:
    """Return (matches_expectation, reason)."""
    if cohort == "opioid_ed":
        if n_drugs <= 0 or n_icds <= 0 or n_cpts <= 0:
            return False, (
                f"metadata expected Drug+ICD+CPT (all >0), got drugs={n_drugs} icds={n_icds} cpts={n_cpts}"
            )
        if n_icd_f <= 0 and n_cpt_f <= 0:
            return False, (
                f"combined_importance expected ICD/CPT features, got drug={n_drug_f} icd={n_icd_f} cpt={n_cpt_f}"
            )
        return True, "Drug+ICD+CPT"

    if cohort == "non_opioid_ed":
        if n_drugs <= 0:
            return False, f"metadata expected Drug only (drugs>0), got drugs={n_drugs}"
        if n_icds != 0 or n_cpts != 0:
            return False, (
                f"metadata expected Drug only (icds=0,cpts=0), got icds={n_icds} cpts={n_cpts}"
            )
        if n_icd_f > 0 or n_cpt_f > 0:
            return False, (
                f"combined_importance expected drug-only, got drug={n_drug_f} icd={n_icd_f} cpt={n_cpt_f}"
            )
        return True, "Drug only"

    return True, "no rule (cohort not in expectations)"

def read_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)

print("Combined SHAP/FFA and metadata review (by cohort)")
print("Expected: opioid_ed = Drug + ICD + CPT; non_opioid_ed = Drug only")
print("=" * 90)

all_ok = True

for cohort, bands in REQUIRED_COHORTS.items():
    meta_path = META_DIR / f"metadata_{cohort}.json"
    meta_exists = meta_path.exists()
    meta = read_json(meta_path) if meta_exists else {}

    for age_band in bands:
        ab = age_band.replace("-", "_")

        codes = meta.get("codes", {}).get(age_band, {"drugs": [], "icds": [], "cpts": []})
        n_drugs = len(codes.get("drugs", []) or [])
        n_icds  = len(codes.get("icds", []) or [])
        n_cpts  = len(codes.get("cpts", []) or [])

        combined_path = CAUSAL_VISUALS / cohort / ab / "combined_importance.csv"
        if not combined_path.exists():
            combined_path = DASHBOARD_OUT / cohort / ab / "combined_importance.csv"
        combined_exists = combined_path.exists()

        n_features = 0
        cols = []
        n_drug_f = n_icd_f = n_cpt_f = 0

        if combined_exists:
            df = pd.read_csv(combined_path)
            cols = list(df.columns)
            n_features = len(df)

            if "feature" in df.columns:
                feats = df["feature"].astype(str)
                n_drug_f = int(feats.str.startswith(DRUG_PREFIX, na=False).sum())
                n_icd_f  = int(feats.str.startswith(ICD_PREFIX,  na=False).sum())
                n_cpt_f  = int(feats.str.startswith(CPT_PREFIX,  na=False).sum())
            else:
                n_drug_f = n_icd_f = n_cpt_f = 0

        ok, reason = check_cohort_expectations(
            cohort=cohort,
            n_drugs=n_drugs, n_icds=n_icds, n_cpts=n_cpts,
            n_drug_f=n_drug_f, n_icd_f=n_icd_f, n_cpt_f=n_cpt_f
        )

        all_ok = all_ok and ok
        status = "OK" if ok else "FAIL"

        print(
            f"{cohort:14s}  age_band={age_band:7s}  "
            f"meta(drug/icd/cpt)={n_drugs:4d}/{n_icds:4d}/{n_cpts:4d}  "
            f"combined_exists={str(combined_exists):5s}  "
            f"combined(drug/icd/cpt)={n_drug_f:4d}/{n_icd_f:4d}/{n_cpt_f:4d}  "
            f"status={status:4s}  {reason}"
        )

        if not meta_exists:
            print(f"  NOTE: metadata file missing: {meta_path}")
        if not combined_exists:
            print(f"  NOTE: combined_importance missing: {combined_path}")

print("=" * 90)
print("Overall:", "OK" if all_ok else "FAIL")

Combined SHAP/FFA and metadata review (by cohort)
Expected: opioid_ed = Drug + ICD + CPT; non_opioid_ed = Drug only
opioid_ed       age_band=0-12     meta(drug/icd/cpt)=  34/  60/ 106  combined_exists=True   combined(drug/icd/cpt)=   1/   8/  43  status=OK    Drug+ICD+CPT
opioid_ed       age_band=13-24    meta(drug/icd/cpt)=  39/  52/ 109  combined_exists=True   combined(drug/icd/cpt)=  48/ 131/ 201  status=OK    Drug+ICD+CPT
opioid_ed       age_band=25-44    meta(drug/icd/cpt)=  44/  67/  89  combined_exists=True   combined(drug/icd/cpt)=  95/ 266/ 315  status=OK    Drug+ICD+CPT
opioid_ed       age_band=45-54    meta(drug/icd/cpt)=  49/  69/  82  combined_exists=True   combined(drug/icd/cpt)=  97/ 203/ 289  status=OK    Drug+ICD+CPT
opioid_ed       age_band=55-64    meta(drug/icd/cpt)=  65/  56/  79  combined_exists=True   combined(drug/icd/cpt)= 124/ 238/ 299  status=OK    Drug+ICD+CPT
opioid_ed       age_band=65-74    meta(drug/icd/cpt)=  46/  78/  76  combined_exists=True   combine

# Shutdown EC2

In [ ]:
SHUTDOWN_EC2 = True  # Set to False to disable auto-shutdown

print("=" * 80)
print("Final Step: EC2 Instance Shutdown (Optional)")
print("=" * 80)

if SHUTDOWN_EC2:
    print("\nShutting down EC2 instance...")
    print("-" * 80)

    import subprocess
    import shutil
    import os

    try:
        # Retrieve EC2 instance ID from metadata service
        result = subprocess.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True,
            text=True,
            timeout=5
        )

        instance_id = result.stdout.strip()

        if instance_id:
            print(f"Instance ID: {instance_id}")

            # Locate AWS CLI
            aws_cmd = shutil.which("aws")
            if not aws_cmd:
                for path in [
                    "/usr/local/bin/aws",
                    "/usr/bin/aws",
                    "/home/ec2-user/.local/bin/aws"
                ]:
                    if os.path.exists(path):
                        aws_cmd = path
                        break

            if not aws_cmd:
                print("\nWarning: AWS CLI not found. Cannot stop instance.")
                print("Install AWS CLI or ensure it is in your PATH.")
                logger.warning("AWS CLI not found; cannot stop EC2 instance")
            else:
                shutdown_cmd = [
                    aws_cmd,
                    "ec2",
                    "stop-instances",
                    "--instance-ids",
                    instance_id
                ]

                print(f"Running: {' '.join(shutdown_cmd)}")
                result = subprocess.run(
                    shutdown_cmd,
                    capture_output=True,
                    text=True
                )

                if result.returncode == 0:
                    print("\nEC2 stop command sent successfully.")
                    print("Instance will stop shortly.")
                    print("Note: This is a STOP (not terminate).")
                    logger.info(
                        f"EC2 instance {instance_id} stop command issued"
                    )
                else:
                    print(
                        f"\nWarning: EC2 stop command failed "
                        f"(exit code {result.returncode})"
                    )
                    if result.stderr:
                        print(f"Error: {result.stderr.strip()}")
                    logger.warning(
                        f"EC2 stop command failed: {result.stderr}"
                    )
        else:
            print("\nWarning: Instance ID not found. Skipping shutdown.")
            print("Manual shutdown command:")
            print("  aws ec2 stop-instances --instance-ids <instance-id>")
            logger.warning("EC2 instance ID could not be determined")

    except subprocess.TimeoutExpired:
        print("\nWarning: Timeout contacting EC2 metadata service.")
        logger.warning("Timeout retrieving EC2 instance ID")

    except Exception as e:
        print(f"\nWarning: Error during EC2 shutdown: {e}")
        logger.warning(f"EC2 shutdown exception: {e}")

else:
    print("\nEC2 Auto-Shutdown: DISABLED")
    print("Set SHUTDOWN_EC2 = True to enable it.")

print("\n" + "=" * 80)
print("Workflow Complete!")
print("=" * 80)

